In [75]:
import pandas as pd

df=pd.read_csv("heart_disease.csv")

df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


In [76]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  target    303 non-null    int64  
dtypes: float64(3), int64(11)
memory usage: 33.3 KB
None


In [77]:
print(df.describe())

              age         sex          cp  ...          ca        thal      target
count  303.000000  303.000000  303.000000  ...  299.000000  301.000000  303.000000
mean    54.438944    0.679868    3.158416  ...    0.672241    4.734219    0.458746
std      9.038662    0.467299    0.960126  ...    0.937438    1.939706    0.499120
min     29.000000    0.000000    1.000000  ...    0.000000    3.000000    0.000000
25%     48.000000    0.000000    3.000000  ...    0.000000    3.000000    0.000000
50%     56.000000    1.000000    3.000000  ...    0.000000    3.000000    0.000000
75%     61.000000    1.000000    4.000000  ...    1.000000    7.000000    1.000000
max     77.000000    1.000000    4.000000  ...    3.000000    7.000000    1.000000

[8 rows x 14 columns]


In [78]:
print(df.isnull().sum())

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
target      0
dtype: int64


In [79]:
print(df["target"].value_counts())

target
0    164
1    139
Name: count, dtype: int64


In [80]:
from sklearn.model_selection import train_test_split
x=df.drop("target",axis=1)
y=df["target"]

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [81]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

lr_pipeline=Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

dt_pipeline=Pipeline([
    ("impute",SimpleImputer(strategy="most_frequent")),
    ("model",DecisionTreeClassifier(max_depth=5,random_state=42))
])

rf_pipeline=Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("model",RandomForestClassifier(n_estimators=100, random_state=42))
])


In [82]:
lr_pipeline.fit(x_train, y_train)
dt_pipeline.fit(x_train, y_train)
rf_pipeline.fit(x_train, y_train)

y_pred_lr = lr_pipeline.predict(x_test)
y_pred_dt = dt_pipeline.predict(x_test)
y_pred_rf = rf_pipeline.predict(x_test)

lr_probs = lr_pipeline.predict_proba(x_test)[:, 1]
dt_probs = dt_pipeline.predict_proba(x_test)[:, 1]
rf_probs = rf_pipeline.predict_proba(x_test)[:, 1]


In [83]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score

results = pd.DataFrame(
    {
        "Model": ["Logistic Regression", "Decision Tree", "Random Forest"],
        "Accuracy": [
            accuracy_score(y_test, y_pred_lr),
            accuracy_score(y_test, y_pred_dt),
            accuracy_score(y_test, y_pred_rf),
        ],
        "ROC-AUC": [
            roc_auc_score(y_test, lr_probs),
            roc_auc_score(y_test, dt_probs),
            roc_auc_score(y_test, rf_probs),
        ],
        "Precision": [
            precision_score(y_test, y_pred_lr),
            precision_score(y_test, y_pred_dt),
            precision_score(y_test, y_pred_rf),
        ],
        "Recall":[
            recall_score(y_test,y_pred_lr),
            recall_score(y_test,y_pred_dt),
            recall_score(y_test,y_pred_rf),
        ],
        "F1 score":[
          f1_score(y_test,y_pred_lr),
          f1_score(y_test,y_pred_dt),
          f1_score(y_test,y_pred_rf),  
        ]
    }
)

print(results)

                 Model  Accuracy   ROC-AUC  Precision    Recall  F1 score
0  Logistic Regression  0.868852  0.951299   0.812500  0.928571  0.866667
1        Decision Tree  0.786885  0.804654   0.727273  0.857143  0.786885
2        Random Forest  0.885246  0.951840   0.838710  0.928571  0.881356


In [84]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

lr_cv=cross_validate(lr_pipeline, x_train,y_train,cv=cv,scoring=scoring)
dt_cv=cross_validate(dt_pipeline,x_train,y_train,cv=cv, scoring=scoring)
rf_cv=cross_validate(rf_pipeline,x_train,y_train,cv=cv, scoring=scoring)

In [85]:
def show_cv_results(name, cv_results):

    print(name,":")
    for metric in scoring:

        scores = cv_results["test_" + metric]

        print(
            f"{metric}: "
            f"Mean = {scores.mean():.4f}, "
            f"Std = {scores.std():.4f}"
        )


show_cv_results("Logistic Regression", lr_cv)
show_cv_results("Decision Tree", dt_cv)
show_cv_results("Random Forest", rf_cv)

Logistic Regression :
accuracy: Mean = 0.8264, Std = 0.0172
precision: Mean = 0.8441, Std = 0.0337
recall: Mean = 0.7652, Std = 0.0547
f1: Mean = 0.8007, Std = 0.0260
roc_auc: Mean = 0.8960, Std = 0.0148
Decision Tree :
accuracy: Mean = 0.7395, Std = 0.0398
precision: Mean = 0.7341, Std = 0.0409
recall: Mean = 0.6751, Std = 0.0800
f1: Mean = 0.7018, Std = 0.0569
roc_auc: Mean = 0.7442, Std = 0.0513
Random Forest :
accuracy: Mean = 0.8016, Std = 0.0284
precision: Mean = 0.8180, Std = 0.0772
recall: Mean = 0.7470, Std = 0.0808
f1: Mean = 0.7746, Std = 0.0326
roc_auc: Mean = 0.8806, Std = 0.0325
